In [1]:
import pandas as pd
import numpy as np

In [2]:
import seaborn as sns
import matplotlib.pyplot as plt

In [3]:
boundaries_df = pd.read_csv("/scratch1/smaruj/boundary_disruption_data.tsv", sep="\t")

In [5]:
len(boundaries_df.groupby(['chrom', 'window_start', 'window_end']))

4474

In [ ]:
4474 - 3915

In [6]:
# restricting the window to +-28kb
window_types = [f"down{i}" for i in range(12,0,-1)] + [f"tad{i}" for i in range(1,6)] + [f"up{i}" for i in range(1,13)]
boundaries_df = boundaries_df[boundaries_df['type'].isin(window_types)].reset_index(drop=True)

In [7]:
len(boundaries_df)

129746

In [8]:
# order of disrupted bins from downstream to upstream
type_order = [f"down{i}" for i in range(12,0,-1)] + [f"tad{i}" for i in range(1,6)] + [f"up{i}" for i in range(1,13)]

In [ ]:
# Set up the figure
plt.figure(figsize=(14, 6))

# Create the violin plot
sns.violinplot(data=boundaries_df, x='type', y='SCD_h1_m0_t0', inner='box', order=type_order)
# target 1 is Bonev ES Cells, which HiC TAD boundaries have called on

# Add labels and title
plt.xlabel('Bin')
plt.ylabel('Disruption Score (target 0, Hsieh)')
plt.title('Disruption Scores by Disrupted Bin')
plt.xticks(rotation=90)

# Show the plot
plt.show()

In [ ]:
# Randomly sample 100 unique (chrom, start, end) regions
sampled_df = boundaries_df.drop_duplicates(subset=['chrom', 'start', 'end']).sample(n=5)

# Filter the original dataframe to include only the sampled regions
filtered_df = boundaries_df[boundaries_df[['chrom', 'start', 'end']].apply(tuple, axis=1).isin(sampled_df[['chrom', 'start', 'end']].apply(tuple, axis=1))]

# Set up the figure
plt.figure(figsize=(16, 8))

# Create a mapping from type to order
type_mapping = {t: i for i, t in enumerate(type_order)}

# For each sampled region, plot the line connecting SCD values for each type
for (chrom, start, end), group in filtered_df.groupby(['chrom', 'start', 'end']):
    # Sort the group by the defined type order
    group = group.sort_values(by='type', key=lambda x: x.map(type_mapping))
    plt.plot(group['type'], group['SCD_h1_m0_t1'], marker='o', label=f"{chrom}:{start}-{end}", alpha=0.6)

# Add labels and title
plt.xlabel('Bin')
plt.ylabel('Disruption Score')
plt.title('Disruption Score Across Bins for 5 Random (chrom, start, end) Regions')

plt.xticks(rotation=45)  # Rotate x-axis labels for better readability
plt.tight_layout()

# Show the plot
plt.show()

In [ ]:
from skimage.filters import threshold_li

In [ ]:
thresholds_li = threshold_li(boundaries_df['SCD_h1_m0_t0'].values.astype('float32'))

In [ ]:
thresholds_li

In [ ]:
plt.figure(figsize=(10, 6))

# Plotting the histogram for explained_tads
plt.hist(boundaries_df['SCD_h1_m0_t0'], bins=50, alpha=0.5, color='green', edgecolor='black')

# thresholds_li = threshold_li(result_df['SCD_h1_m0_t1'].values)

plt.axvline(thresholds_li, color='red', linestyle='dashed', linewidth=2, label='Li Threshold')

# # Adding labels and title
plt.xlabel('Disruption Score')
plt.ylabel('Frequency')
plt.title('Histogram of Maximum Disruption Score for Tested TADs')
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.7)

# plt.savefig("./plots/max_disruption_threshold.pdf", format="pdf")

# Show the plot
plt.show()

In [ ]:
# Randomly sample 100 unique (chrom, start, end) regions
sampled_df = boundaries_df.drop_duplicates(subset=['chrom', 'start', 'end']).sample(n=5)

# Filter the original dataframe to include only the sampled regions
filtered_df = boundaries_df[boundaries_df[['chrom', 'start', 'end']].apply(tuple, axis=1).isin(sampled_df[['chrom', 'start', 'end']].apply(tuple, axis=1))]

# Set up the figure
plt.figure(figsize=(16, 8))

# Create a mapping from type to order
type_mapping = {t: i for i, t in enumerate(type_order)}

# For each sampled region, plot the line connecting SCD values for each type
for (chrom, start, end), group in filtered_df.groupby(['chrom', 'start', 'end']):
    # Sort the group by the defined type order
    group = group.sort_values(by='type', key=lambda x: x.map(type_mapping))
    plt.plot(group['type'], group['SCD_h1_m0_t1'], marker='o', label=f"{chrom}:{start}-{end}", alpha=0.6)

# Add labels and title
plt.xlabel('Bin')
plt.ylabel('Disruption Score')
plt.title('Disruption Score Across Bins for 5 Random (chrom, start, end) Regions')

plt.axhline(y=thresholds_li, color='red', linestyle='--', linewidth=1, label='Li threshold')
plt.legend()
plt.xticks(rotation=45)  # Rotate x-axis labels for better readability
plt.tight_layout()

# Show the plot
plt.show()

In [ ]:
sensitive_bins = boundaries_df[boundaries_df["SCD_h1_m0_t0"] > thresholds_li]

In [12]:
disr_bins = [i for i in range(241, 270)]
# boundary overlapping bins are: 253, 254, 255, 256, 257

In [ ]:
type_to_num = dict(zip(type_order, disr_bins))
sensitive_bins['disrupted_bin'] = sensitive_bins['type'].map(type_to_num)

In [ ]:
sensitive_bins = sensitive_bins.reset_index(drop=True)

In [ ]:
sensitive_bins = sensitive_bins.drop(columns=["rel_disruption_end", "rel_disruption_start", "type"])

In [ ]:
sensitive_bins = sensitive_bins.drop(columns=["SCD_h1_m0_t0", "SCD_h1_m0_t1", "SCD_h1_m0_t2", "SCD_h1_m0_t3", "SCD_h1_m0_t4", "SCD_h1_m0_t5"])

In [ ]:
grouped_df = (
    sensitive_bins
    .groupby(["chrom", "start", "end", "window_end", "window_start"])["disrupted_bin"]
    .apply(list)
    .reset_index()
)

In [ ]:
grouped_df

In [ ]:
# grouped_df.to_csv("/scratch1/smaruj/sensitive_bins_boundaries.tsv", sep="\t", index=False)

In [9]:
# Group by boundary (window_start, window_end) and get the index of the row with the highest disruption
idx = boundaries_df.groupby(['chrom', 'window_start', 'window_end'])['SCD_h1_m0_t0'].idxmax()

# Use the indices to select the corresponding rows
df_max_disruption = boundaries_df.loc[idx].reset_index(drop=True)

In [10]:
df_max_disruption

,SCD_h1_m0_t0,SCD_h1_m0_t1,SCD_h1_m0_t2,SCD_h1_m0_t3,SCD_h1_m0_t4,SCD_h1_m0_t5,chrom,end,rel_disruption_end,rel_disruption_start,start,type,window_end,window_start
0,12.625,13.98,9.84,9.760,10.56,8.800,chr1,4410000,667648,665600,4400000,up4,5061504,3750784
1,39.750,45.53,46.88,47.340,45.80,40.400,chr1,4780000,649216,647168,4770000,down1,5431504,4120784
2,18.620,21.22,15.00,14.470,16.05,13.266,chr1,5160000,684032,681984,5150000,up12,5811504,4500784
3,38.250,49.34,53.12,48.970,49.88,41.120,chr1,5910000,655360,653312,5900000,tad3,6561504,5250784
4,12.480,14.41,14.84,15.160,13.95,11.836,chr1,6200000,651264,649216,6190000,tad1,6851504,5540784
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4469,13.080,14.07,22.34,22.400,19.30,17.110,chr9,121460000,679936,677888,121450000,up10,122111504,120800784
4470,9.890,9.09,10.04,10.080,9.66,8.310,chr9,121710000,647168,645120,121700000,down2,122361504,121050784
4471,8.640,8.55,8.88,8.586,8.47,7.280,chr9,122360000,651264,649216,122350000,tad1,123011504,121700784
4472,33.400,36.03,49.88,50.530,45.03,39.880,chr9,122730000,667648,665600,122720000,up4,123381504,122070784


In [13]:
type_to_num = dict(zip(type_order, disr_bins))
df_max_disruption['disrupted_bin'] = df_max_disruption['type'].map(type_to_num)

In [15]:
df_max_disruption.to_csv("/scratch1/smaruj/single_bins_boundaries.tsv", sep="\t", index=False)